In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Connect to the database file
# (Make sure the notebook is in the same folder, or provide the full path)
conn = sqlite3.connect('advanced_lab_inventory.db')

# 2. Pull the usage data, joining it with the item names
query = """
    SELECT u.date_used, i.name, u.amount_used, i.unit, u.user_name
    FROM usage_log u
    JOIN inventory i ON u.item_id = i.item_id
"""

# Load it directly into a Pandas DataFrame
df_usage = pd.read_sql_query(query, conn)

# Always close the connection when done fetching!
conn.close()

# 3. Clean up the data for analysis
# Convert the timestamp string into a real datetime object
df_usage['date_used'] = pd.to_datetime(df_usage['date_used'])

# --- Example Analysis: Plotting Usage Over Time ---
# Let's say we want to track how much LB Broth is being used weekly
item_to_track = "LB Broth"

# Filter for the specific item
df_item = df_usage[df_usage['name'] == item_to_track].copy()

if not df_item.empty:
    # Set the date as the index for time-series grouping
    df_item.set_index('date_used', inplace=True)
    
    # Resample the data by week ('W') and sum the amounts
    weekly_usage = df_item['amount_used'].resample('W').sum()
    
    # Plot it
    plt.figure(figsize=(10, 5))
    weekly_usage.plot(kind='bar', color='skyblue')
    plt.title(f"Weekly Usage of {item_to_track}")
    plt.ylabel(f"Amount Used ({df_item['unit'].iloc[0]})")
    plt.xlabel("Week")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print(f"No usage data found for {item_to_track}")